# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [2]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [ ]:

# Task 1 - Comparing prices for room types

# keeping only these 2 room types
filtered_df = df_cap[df_cap['room_type'].isin(['Entire home/apt', 'Private room'])]

# median prices
entire_median = filtered_df[filtered_df['room_type'] == 'Entire home/apt']['price'].median()
private_median = filtered_df[filtered_df['room_type'] == 'Private room']['price'].median()

fig = px.histogram(
    filtered_df,
    x='price',
    color='room_type',
    nbins=45,
    opacity=0.55,
    barmode='overlay',
    title='Price comparison between entire apartments and private rooms',
    color_discrete_map={
        'Entire home/apt': 'teal',
        'Private room': 'orange'
    }
)

fig.add_vline(
    x=entire_median,
    line_dash='dash',
    line_color='teal',
    annotation_text=f'Entire home median = £{entire_median:.0f}'
)

fig.add_vline(
    x=private_median,
    line_dash='dot',
    line_color='orange',
    annotation_text=f'Private room median = £{private_median:.0f}'
)

fig.add_annotation(
    text='Extreme values were capped using 95th percentile',
    xref='paper',
    yref='paper',
    x=1,
    y=1.07,
    showarrow=False
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Price per night (£)',
    yaxis_title='Number of listings'
)

fig.show()



### Observation
Entire homes/apartments are usually more expensive compared to private rooms.  
Most private room listings are concentrated in the lower price range.


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [ ]:

# Task 2 - Reviews per month by borough

# removing missing values
review_df = df_cap.dropna(subset=['reviews_per_month', 'neighbourhood_group'])

borough_rank = (
    review_df.groupby('neighbourhood_group')['reviews_per_month']
    .median()
    .sort_values(ascending=False)
)

top_boroughs = borough_rank.head(2).index.tolist()

review_df['group_type'] = review_df['neighbourhood_group'].apply(
    lambda x: 'Highlighted Boroughs' if x in top_boroughs else 'Other Boroughs'
)

fig = px.box(
    review_df,
    x='reviews_per_month',
    y='neighbourhood_group',
    color='group_type',
    points='outliers',
    category_orders={'neighbourhood_group': borough_rank.index.tolist()},
    title='Monthly review activity across London boroughs',
    color_discrete_map={
        'Highlighted Boroughs': 'crimson',
        'Other Boroughs': 'gray'
    }
)

overall_reviews = review_df['reviews_per_month'].median()

fig.add_vline(
    x=overall_reviews,
    line_dash='dash',
    line_color='black',
    annotation_text=f'Overall median = {overall_reviews:.2f}'
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Reviews per month',
    yaxis_title='Neighbourhood group'
)

fig.show()



### Observation
Some boroughs receive noticeably more reviews per month, which may indicate higher booking activity.  
The highlighted boroughs have a higher median review count than most other areas.
